# Checkpoint Loss Curves

This notebook reads the analysis files generated by `scripts/eval_infilling_checkpoint_losses.py` and plots training loss against checkpoint re-evaluated validation loss for the four APRIL infilling experiments.

Expected default outputs:
- `outputs/analysis/checkpoint_loss_curves/merged_loss_curves.csv`
- `outputs/analysis/checkpoint_loss_curves/run_manifest.json`

If you wrote results somewhere else, update `ANALYSIS_DIR` in the next cell.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "outputs").exists():
    REPO_ROOT = Path("/groups/tensorlab/myang2/cs165-project")

ANALYSIS_DIR = REPO_ROOT / "outputs" / "analysis" / "checkpoint_loss_curves"
MERGED_CSV = ANALYSIS_DIR / "merged_loss_curves.csv"
MANIFEST_JSON = ANALYSIS_DIR / "run_manifest.json"

if not MERGED_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {MERGED_CSV}. Run scripts/eval_infilling_checkpoint_losses.py first."
    )
if not MANIFEST_JSON.exists():
    raise FileNotFoundError(
        f"Could not find {MANIFEST_JSON}. Run scripts/eval_infilling_checkpoint_losses.py first."
    )

loss_df = pd.read_csv(MERGED_CSV)
with MANIFEST_JSON.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

loss_df["epoch"] = pd.to_numeric(loss_df.get("epoch"), errors="coerce")
loss_df["step"] = pd.to_numeric(loss_df.get("step"), errors="coerce")
loss_df["loss_value"] = pd.to_numeric(loss_df.get("loss_value"), errors="coerce")
loss_df["requested_epoch"] = pd.to_numeric(
    loss_df.get("requested_epoch"), errors="coerce"
)

experiment_order = [row["experiment_name"] for row in manifest.get("experiments", [])]
if not experiment_order:
    experiment_order = sorted(loss_df["experiment_name"].dropna().unique())

print(f"Loaded {len(loss_df):,} rows from {MERGED_CSV}")
display(loss_df.head())

In [ ]:
def rows_for_metric(experiment_name: str, metric_name: str) -> pd.DataFrame:
    rows = loss_df[
        (loss_df["experiment_name"] == experiment_name)
        & (loss_df["metric_name"] == metric_name)
    ].copy()
    if metric_name == "checkpoint_val_loss":
        rows = rows[rows["status"] == "evaluated"]
        rows = rows.sort_values(["requested_epoch", "epoch", "step"])
    else:
        rows = rows.sort_values(["step", "epoch"])
    return rows


def missing_epoch_rows(experiment_name: str) -> pd.DataFrame:
    rows = loss_df[
        (loss_df["experiment_name"] == experiment_name)
        & (loss_df["metric_name"] == "checkpoint_val_loss")
        & (loss_df["status"] != "evaluated")
    ].copy()
    return rows.sort_values(["requested_epoch", "epoch"])


def plot_experiment(experiment_name: str, ax=None) -> None:
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))

    train_rows = rows_for_metric(experiment_name, "train_loss")
    checkpoint_val_rows = rows_for_metric(experiment_name, "checkpoint_val_loss")
    logged_val_rows = rows_for_metric(experiment_name, "logged_eval_loss")
    missing_rows = missing_epoch_rows(experiment_name)

    if not train_rows.empty:
        ax.plot(
            train_rows["epoch"],
            train_rows["loss_value"],
            label="train loss",
            color="tab:blue",
            linewidth=1.6,
            alpha=0.9,
        )
    if not checkpoint_val_rows.empty:
        ax.plot(
            checkpoint_val_rows["epoch"],
            checkpoint_val_rows["loss_value"],
            label="checkpoint val loss",
            color="tab:orange",
            marker="o",
            linewidth=1.8,
        )
    if not logged_val_rows.empty:
        ax.plot(
            logged_val_rows["epoch"],
            logged_val_rows["loss_value"],
            label="logged eval loss",
            color="tab:green",
            linestyle="--",
            linewidth=1.2,
            alpha=0.8,
        )

    if not missing_rows.empty:
        missing_epochs = ", ".join(
            str(int(epoch)) for epoch in missing_rows["requested_epoch"].dropna().tolist()
        )
        ax.text(
            0.99,
            0.02,
            f"missing checkpoint epochs: {missing_epochs}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9,
            color="tab:red",
        )

    ax.set_title(experiment_name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.grid(True, alpha=0.25)
    ax.legend()


def checkpoint_summary_table() -> pd.DataFrame:
    rows = []
    for experiment_name in experiment_order:
        train_rows = rows_for_metric(experiment_name, "train_loss")
        checkpoint_val_rows = rows_for_metric(experiment_name, "checkpoint_val_loss")
        missing_rows = missing_epoch_rows(experiment_name)
        rows.append(
            {
                "experiment_name": experiment_name,
                "num_train_points": len(train_rows),
                "num_checkpoint_val_points": len(checkpoint_val_rows),
                "min_checkpoint_epoch": checkpoint_val_rows["epoch"].min()
                if not checkpoint_val_rows.empty
                else None,
                "max_checkpoint_epoch": checkpoint_val_rows["epoch"].max()
                if not checkpoint_val_rows.empty
                else None,
                "missing_requested_epochs": ", ".join(
                    str(int(epoch))
                    for epoch in missing_rows["requested_epoch"].dropna().tolist()
                ),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
summary_df = checkpoint_summary_table()
display(summary_df)

num_experiments = max(1, len(experiment_order))
num_cols = 2
num_rows = (num_experiments + num_cols - 1) // num_cols
fig, axes = plt.subplots(num_rows, num_cols, figsize=(14, 4.5 * num_rows), squeeze=False)

for ax, experiment_name in zip(axes.flatten(), experiment_order):
    plot_experiment(experiment_name, ax=ax)

for ax in axes.flatten()[len(experiment_order):]:
    ax.axis("off")

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for experiment_name in experiment_order:
    train_rows = rows_for_metric(experiment_name, "train_loss")
    checkpoint_val_rows = rows_for_metric(experiment_name, "checkpoint_val_loss")

    if not train_rows.empty:
        axes[0].plot(
            train_rows["epoch"],
            train_rows["loss_value"],
            label=experiment_name,
            linewidth=1.2,
        )
    if not checkpoint_val_rows.empty:
        axes[1].plot(
            checkpoint_val_rows["epoch"],
            checkpoint_val_rows["loss_value"],
            marker="o",
            label=experiment_name,
            linewidth=1.6,
        )

axes[0].set_title("Training Loss Across Experiments")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].grid(True, alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].set_title("Checkpoint Validation Loss Across Experiments")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("loss")
axes[1].grid(True, alpha=0.25)
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()